# Grupo 1: Data Governance y Calidad de Datos
## Validación de Calidad de Datos con Great Expectations
### Dataset: Titanic (Kaggle)

**Demo práctica para exposición** - Implementación de validaciones de calidad de datos usando Great Expectations.

**Framework:** Great Expectations v1.17.0
**Dataset:** Titanic - https://github.com/datasciencedojo/datasets

## ¿Qué es Great Expectations?

Great Expectations (GX) es un framework open-source de Python para validar, documentar y monitorear la calidad de datos. Se basa en el concepto de **expectations** (expectativas), que son afirmaciones verificables sobre cómo debería lucir un dataset.

**Dimensiones de calidad que evaluaremos:**
1. **Integridad (Integrity)** - Valores nulos y unicidad
2. **Exactitud (Accuracy)** - Valores dentro de rangos válidos
3. **Consistencia (Consistency)** - Valores en dominios esperados
4. **Completitud (Completeness)** - Campos obligatorios presentes
5. **Validez (Validity)** - Formatos y longitudes correctas

## Paso 1: Importar librerías y cargar datos

In [1]:
import pandas as pd
import great_expectations as gx
import great_expectations.expectations as gxe
import json
import os
import shutil
import warnings

warnings.filterwarnings("ignore")

PROJECT_DIR = os.getcwd()
DATA_PATH = os.path.join(PROJECT_DIR, "titanic.csv")
GX_DIR = os.path.join(PROJECT_DIR, "great_expectations")

print(f"Great Expectations v{gx.__version__}")
print(f"Pandas v{pd.__version__}")

Great Expectations v1.17.2
Pandas v3.0.3


In [2]:
# Descargar dataset de Titanic
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(url)
df.to_csv(DATA_PATH, index=False)

print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
print(f"Columnas: {list(df.columns)}")

Dataset cargado: 891 filas, 12 columnas
Columnas: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


## Paso 2: Exploración inicial de calidad de datos

Antes de definir expectativas, exploramos los datos para identificar posibles problemas de calidad.

In [3]:
print("=" * 60)
print("EXPLORACIÓN INICIAL DE CALIDAD DE DATOS")
print("=" * 60)

# Vista previa del dataset
df.head()

EXPLORACIÓN INICIAL DE CALIDAD DE DATOS


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
# Valores nulos por columna
print("Valores nulos por columna:")
nulls = df.isnull().sum()
for col, count in nulls.items():
    if count > 0:
        pct = (count / len(df)) * 100
        print(f"  - {col}: {count} nulos ({pct:.1f}%)")

print()

# Estadísticas descriptivas
print("Estadísticas clave:")
print(f"  Edad: min={df['Age'].min():.1f}, max={df['Age'].max():.1f}, mean={df['Age'].mean():.1f}")
print(f"  Tarifa: min=${df['Fare'].min():.2f}, max=${df['Fare'].max():.2f}, mean=${df['Fare'].mean():.2f}")
print(f"  Sobrevivientes: {int(df['Survived'].sum())} de {len(df)} ({df['Survived'].mean()*100:.1f}%)")
print(f"  Duplicados: {df.duplicated().sum()}")

Valores nulos por columna:
  - Age: 177 nulos (19.9%)
  - Cabin: 687 nulos (77.1%)
  - Embarked: 2 nulos (0.2%)

Estadísticas clave:
  Edad: min=0.4, max=80.0, mean=29.7
  Tarifa: min=$0.00, max=$512.33, mean=$32.20
  Sobrevivientes: 342 de 891 (38.4%)
  Duplicados: 0


## Paso 3: Configurar Great Expectations

Inicializamos el contexto de GX, creamos un datasource Pandas y un data asset para nuestro dataset.

In [5]:
# Limpiar directorio GX previo si existe
if os.path.exists(GX_DIR):
    shutil.rmtree(GX_DIR)

# Inicializar contexto de Great Expectations
context = gx.get_context(project_root_dir=GX_DIR)

# Crear datasource pandas y data asset
datasource = context.data_sources.add_pandas("titanic_datasource")
data_asset = datasource.add_dataframe_asset(name="titanic_data")

print(f"Contexto GX inicializado en: {GX_DIR}")
print(f"Datasource: {datasource.name}")
print(f"Data Asset: {data_asset.name}")

Contexto GX inicializado en: /home/asdf/Downloads/DataGover/datavalidation/great_expectations
Datasource: titanic_datasource
Data Asset: titanic_data


## Paso 4: Definir Expectativas de Calidad de Datos

Las **expectativas** son reglas de validación que definen cómo deberían lucir nuestros datos. Cada expectativa verifica una dimensión específica de calidad.

In [6]:
# Crear Expectation Suite (contenedor de expectativas)
suite = context.suites.add(gx.ExpectationSuite(name="titanic_data_quality"))

# =============================================
# EXPECTATIVAS DE CALIDAD DE DATOS
# =============================================

# --- 1. ESTRUCTURA DEL ESQUEMA ---
suite.add_expectation(
    gxe.ExpectTableColumnsToMatchOrderedList(column_list=list(df.columns))
)

# --- 2. COMPLETITUD (valores no nulos en campos obligatorios) ---
suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column="PassengerId"))
suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column="Survived"))
suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column="Name"))
suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column="Sex"))
suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column="Pclass"))
suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column="Fare"))

# --- 3. INTEGRIDAD DE DOMINIOS (valores válidos) ---
suite.add_expectation(gxe.ExpectColumnValuesToBeInSet(column="Survived", value_set=[0, 1]))
suite.add_expectation(gxe.ExpectColumnValuesToBeInSet(column="Sex", value_set=["male", "female"]))
suite.add_expectation(gxe.ExpectColumnValuesToBeInSet(column="Pclass", value_set=[1, 2, 3]))
suite.add_expectation(gxe.ExpectColumnValuesToBeInSet(column="Embarked", value_set=["S", "C", "Q"]))

# --- 4. EXACTITUD (rangos válidos) ---
suite.add_expectation(gxe.ExpectColumnValuesToBeBetween(column="Age", min_value=0, max_value=100, mostly=0.95))
suite.add_expectation(gxe.ExpectColumnValuesToBeBetween(column="Fare", min_value=0, max_value=600))
suite.add_expectation(gxe.ExpectColumnValuesToBeBetween(column="SibSp", min_value=0, max_value=10))
suite.add_expectation(gxe.ExpectColumnValuesToBeBetween(column="Parch", min_value=0, max_value=10))

# --- 5. UNICIDAD ---
suite.add_expectation(gxe.ExpectColumnValuesToBeUnique(column="PassengerId"))

# --- 6. VALIDEZ (longitud de strings) ---
suite.add_expectation(gxe.ExpectColumnValueLengthsToBeBetween(column="Name", min_value=2, max_value=100))
suite.add_expectation(gxe.ExpectColumnValueLengthsToBeBetween(column="Cabin", min_value=1, max_value=10))

# --- 7. ESTADÍSTICAS AVANZADAS ---
suite.add_expectation(gxe.ExpectColumnMeanToBeBetween(column="Age", min_value=25, max_value=35))
suite.add_expectation(gxe.ExpectColumnMedianToBeBetween(column="Age", min_value=25, max_value=32))
suite.add_expectation(gxe.ExpectColumnStdevToBeBetween(column="Age", min_value=10, max_value=20))

expectations_list = [
    ("Estructura", "expect_table_columns_to_match_ordered_list", "table"),
    ("Completitud", "expect_column_values_to_not_be_null", "PassengerId"),
    ("Completitud", "expect_column_values_to_not_be_null", "Survived"),
    ("Completitud", "expect_column_values_to_not_be_null", "Name"),
    ("Completitud", "expect_column_values_to_not_be_null", "Sex"),
    ("Completitud", "expect_column_values_to_not_be_null", "Pclass"),
    ("Completitud", "expect_column_values_to_not_be_null", "Fare"),
    ("Consistencia", "expect_column_values_to_be_in_set", "Survived"),
    ("Consistencia", "expect_column_values_to_be_in_set", "Sex"),
    ("Consistencia", "expect_column_values_to_be_in_set", "Pclass"),
    ("Consistencia", "expect_column_values_to_be_in_set", "Embarked"),
    ("Exactitud", "expect_column_values_to_be_between", "Age"),
    ("Exactitud", "expect_column_values_to_be_between", "Fare"),
    ("Exactitud", "expect_column_values_to_be_between", "SibSp"),
    ("Exactitud", "expect_column_values_to_be_between", "Parch"),
    ("Unicidad", "expect_column_values_to_be_unique", "PassengerId"),
    ("Validez", "expect_column_value_lengths_to_be_between", "Name"),
    ("Validez", "expect_column_value_lengths_to_be_between", "Cabin"),
    ("Estadísticas", "expect_column_mean_to_be_between", "Age"),
    ("Estadísticas", "expect_column_median_to_be_between", "Age"),
    ("Estadísticas", "expect_column_stdev_to_be_between", "Age"),
]

print(f"Expectation Suite creada: '{suite.name}'")
print(f"Total expectativas definidas: {len(expectations_list)}")
print()
print("Lista de expectativas:")
for i, (dim, exp_type, col) in enumerate(expectations_list, 1):
    print(f"  {i:2d}. [{dim:14s}] {exp_type} -> {col}")

Expectation Suite creada: 'titanic_data_quality'
Total expectativas definidas: 21

Lista de expectativas:
   1. [Estructura    ] expect_table_columns_to_match_ordered_list -> table
   2. [Completitud   ] expect_column_values_to_not_be_null -> PassengerId
   3. [Completitud   ] expect_column_values_to_not_be_null -> Survived
   4. [Completitud   ] expect_column_values_to_not_be_null -> Name
   5. [Completitud   ] expect_column_values_to_not_be_null -> Sex
   6. [Completitud   ] expect_column_values_to_not_be_null -> Pclass
   7. [Completitud   ] expect_column_values_to_not_be_null -> Fare
   8. [Consistencia  ] expect_column_values_to_be_in_set -> Survived
   9. [Consistencia  ] expect_column_values_to_be_in_set -> Sex
  10. [Consistencia  ] expect_column_values_to_be_in_set -> Pclass
  11. [Consistencia  ] expect_column_values_to_be_in_set -> Embarked
  12. [Exactitud     ] expect_column_values_to_be_between -> Age
  13. [Exactitud     ] expect_column_values_to_be_between -> Fare
  14.

## Paso 5: Ejecutar Validación

Ejecutamos todas las expectativas contra el dataset y analizamos los resultados.

In [7]:
# Crear batch request con el dataframe
batch_request = data_asset.build_batch_request(options={"dataframe": df})
batch = data_asset.get_batch(batch_request)

# Obtener validador
validator = context.get_validator(batch=batch, expectation_suite=suite)

# Ejecutar validación
results = validator.validate()

# Calcular métricas
total = len(results["results"])
passed = sum(1 for r in results["results"] if r["success"])
failed = total - passed
success_pct = (passed / total) * 100

print("=" * 60)
print("RESULTADOS DE VALIDACIÓN")
print("=" * 60)
print(f"Total expectativas:  {total}")
print(f"Exitosas:            {passed}")
print(f"Fallidas:            {failed}")
print(f"Tasa de éxito:       {success_pct:.1f}%")
print(f"Estado:              {'PASSED' if success_pct >= 90 else 'NEEDS ATTENTION'}")
print("=" * 60)

Calculating Metrics: 100%|██████████| 78/78 [00:00<00:00, 2518.11it/s]

RESULTADOS DE VALIDACIÓN
Total expectativas:  21
Exitosas:            20
Fallidas:            1
Tasa de éxito:       95.2%
Estado:              PASSED


## Paso 6: Análisis de Resultados Fallidos

Las expectativas fallidas son oportunidades de mejora en la calidad de datos. Identifiquemos qué encontró GX.

In [8]:
failed_exps = [r for r in results["results"] if not r["success"]]

if failed_exps:
    print("EXPECTATIVAS FALLIDAS (problemas de calidad detectados):")
    print("-" * 60)
    for r in failed_exps:
        exp_type = r["expectation_config"]["type"]
        col = r["expectation_config"]["kwargs"].get("column", "N/A")
        print(f"Columna: {col}")
        print(f"Expectativa: {exp_type}")
        if "result" in r:
            res = r["result"]
            if "unexpected_count" in res:
                print(f"  Valores inesperados: {res['unexpected_count']}")
            if "unexpected_percent" in res:
                print(f"  Porcentaje inesperado: {res['unexpected_percent']:.1f}%")
            if "observed_value" in res:
                print(f"  Valor observado: {res['observed_value']}")
        print()
else:
    print("Todas las expectativas pasaron exitosamente! ✓")

EXPECTATIVAS FALLIDAS (problemas de calidad detectados):
------------------------------------------------------------
Columna: Cabin
Expectativa: expect_column_value_lengths_to_be_between
  Valores inesperados: 8
  Porcentaje inesperado: 3.9%



## Paso 7: Generar Reporte de Calidad

In [9]:
report_path = os.path.join(PROJECT_DIR, "validation_report.json")
report = {
    "grupo": "Grupo 1: Data Governance y Calidad de Datos",
    "framework": f"Great Expectations v{gx.__version__}",
    "dataset": "Titanic (Kaggle)",
    "total_expectations": len(results["results"]),
    "passed": passed,
    "failed": failed,
    "success_rate": round(success_pct, 1),
    "failed_details": []
}

for r in failed_exps:
    report["failed_details"].append({
        "expectation": r["expectation_config"]["type"],
        "column": r["expectation_config"]["kwargs"].get("column", "N/A"),
        "unexpected_count": r.get("result", {}).get("unexpected_count", 0),
        "unexpected_percent": r.get("result", {}).get("unexpected_percent", 0)
    })

with open(report_path, "w") as f:
    json.dump(report, f, indent=2)

print("REPORTE GENERADO")
print("=" * 60)
print(json.dumps(report, indent=2))
print(f"\nReporte guardado en: {report_path}")

REPORTE GENERADO
{
  "grupo": "Grupo 1: Data Governance y Calidad de Datos",
  "framework": "Great Expectations v1.17.2",
  "dataset": "Titanic (Kaggle)",
  "total_expectations": 21,
  "passed": 20,
  "failed": 1,
  "success_rate": 95.2,
  "failed_details": [
    {
      "expectation": "expect_column_value_lengths_to_be_between",
      "column": "Cabin",
      "unexpected_count": 8,
      "unexpected_percent": 3.9215686274509802
    }
  ]
}

Reporte guardado en: /home/asdf/Downloads/DataGover/datavalidation/validation_report.json


## Paso 8: Resumen de la Demo

### ¿Qué demostramos?

| Dimensión | Expectativas | Resultado |
|-----------|-------------|-----------|
| **Estructura** | 1 - Esquma completo | ✓ |
| **Completitud** | 6 - Campos no nulos | ✓ |
| **Consistencia** | 4 - Dominios válidos | ✓ |
| **Exactitud** | 4 - Rangos válidos | ✓ |
| **Unicidad** | 1 - IDs únicos | ✓ |
| **Validez** | 2 - Longitud strings | ⚠ 1 falla |
| **Estadísticas** | 3 - Mean, median, stdev | ✓ |

### Hallazgo clave para la exposición

Great Expectations detectó que la columna **Cabin** tiene valores con longitudes fuera del rango esperado (1-10 caracteres). Esto es un ejemplo real de cómo las herramientas de gobernanza de datos identifican problemas de calidad que podrían afectar análisis posteriores.

### Conclusión

Con **21 expectativas** evaluadas y una **tasa de éxito del 95.2%**, el dataset de Titanic muestra buena calidad general, pero con oportunidades de mejora en la columna Cabin. Great Expectations proporciona un framework robusto y automatizable para la validación de datos en pipelines de gobernanza.